In [ ]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
from rank_bm25 import BM25Okapi


In [3]:
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.config import client
from src.text_utils import normalize, tokenize
from scripts.build_index import build_index
from src.retrieval_search import search_bm25, search_faiss

RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
IDX_DIR = ROOT / "data" / "indices"

print(
    "ROOT:",
    ROOT,
    "| RAW exists:",
    RAW_DIR.exists(),
    "| PROC exists:",
    PROC_DIR.exists(),
    "| IDX exists:",
    IDX_DIR.exists(),
)

# Load the files into c and p
c = pd.read_parquet(RAW_DIR / "comments_v0.parquet")
p = pd.read_parquet(RAW_DIR / "products_v0.parquet")

ROOT: c:\Users\98936\Desktop\DigiSmart Assistant | RAW exists: True | PROC exists: True | IDX exists: True


In [5]:
# --- Overview ---
print("Comment columns:", c.columns.tolist())
print("Product columns:", p.columns.tolist())
print("Duplicates in products id:", p["id"].duplicated().sum())
print("Sample product IDs:", p["id"].head(3).tolist())
print("Sample comment product_ids:", c["product_id"].head(3).tolist())


# --- Data types & Null values ---
print("\n=== comments: dtypes & nulls ===")
print(
    pd.DataFrame(
        {
            "dtype": c.dtypes.astype(str),
            "nulls": c.isna().sum(),
            "null_%": (c.isna().mean() * 100).round(1),
        }
    ).to_string()
)


print("\n=== products: dtypes & nulls ===")
print(
    pd.DataFrame(
        {
            "dtype": p.dtypes.astype(str),
            "nulls": p.isna().sum(),
            "null_%": (p.isna().mean() * 100).round(1),
        }
    ).to_string()
)


# --- Numeric summary ---
print("\n=== comments: numeric summary ===")
print(c.describe().T.round(2).to_string())


print("\n=== products: numeric summary ===")
print(p.describe().T.round(2).to_string())


# --- Key value distributions ---
for col in ["rate", "recommendation_status", "is_buyer"]:
    if col in c.columns:
        print(f"\ncomments[{col}] value counts:")
        print(c[col].value_counts(dropna=False).head(10).to_string())


for col in ["Category1", "Brand", "Is_Fake"]:
    if col in p.columns:
        print(f"\nproducts[{col}] value counts:")
        print(p[col].value_counts(dropna=False).head(10).to_string())


# --- Sample rows ---
print("\n=== comments sample ===")
display(c.head(3))
print("\n=== products sample ===")
display(p.head(3))

Comment columns: ['id', 'title', 'body', 'created_at', 'rate', 'recommendation_status', 'is_buyer', 'product_id', 'advantages', 'disadvantages', 'likes', 'dislikes', 'seller_title', 'seller_code', 'true_to_size_rate']
Product columns: ['id', 'title_fa', 'Rate', 'Rate_cnt', 'Category1', 'Category2', 'Brand', 'Price', 'Seller', 'Is_Fake', 'min_price_last_month', 'sub_category']
Duplicates in products id: 76
Sample product IDs: [7096438, 2845119, 6117745]
Sample comment product_ids: [252058, 252058, 3331597]

=== comments: dtypes & nulls ===
                         dtype  nulls  null_%
id                       int64      0     0.0
title                      str      0     0.0
body                       str      0     0.0
created_at                 str      0     0.0
rate                   float64      0     0.0
recommendation_status      str      0     0.0
is_buyer                  bool      0     0.0
product_id               int64      0     0.0
advantages                 str      0    

,id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,likes,dislikes,seller_title,seller_code,true_to_size_rate
0,53672599,پیشنهاد نمیشود,به درد نمیخوره,23 شهریور 1402,1.0,not_recommended,True,252058,nan,nan,0,0,دیجی‌کالا,5A52N,nan
1,9897229,بسته بندی بد,می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بن...,16 تیر 1399,0.0,recommended,True,252058,['تجربه جالبی بود برام '],['بسته بندی جالبی نداشت'],1,0,دیجی‌کالا,5A52N,nan
2,38074516,برس ریمل,بسته بندیش خوب بود\r\n کاربرد و کیفیتشم خیلی خ...,26 مرداد 1401,0.0,recommended,True,3331597,nan,nan,0,0,آرالیا بیوتی,ADM47,nan



=== products sample ===


,id,title_fa,Rate,Rate_cnt,Category1,Category2,Brand,Price,Seller,Is_Fake,min_price_last_month,sub_category
0,7096438,آبسلانگ مدل s5 بسته 250 عددی,90,4,آبسلانگ,NaN,متفرقه,634800,سلامت ساز راد,False,0,beauty
1,2845119,آبسلانگ مدل M-1 بسته 400 عددی,84,217,آبسلانگ,NaN,متفرقه,818800,مهر افزون,False,0,beauty
2,6117745,آبسلانگ مدل m50 مجموعه 500 عددی,88,14,آبسلانگ,NaN,متفرقه,920000,یانگوم,False,0,beauty


In [6]:
# IDs that appear more than once
duplicate_ids = p.loc[p["id"].duplicated(keep=False), "id"]

print("Number of duplicate rows:", p["id"].duplicated().sum())
print("Number of product IDs with duplicates:", duplicate_ids.nunique())

# Show duplicate product rows
duplicates = p[p["id"].isin(duplicate_ids)].sort_values("id")
display(duplicates)

Number of duplicate rows: 76
Number of product IDs with duplicates: 75


,id,title_fa,Rate,Rate_cnt,Category1,Category2,Brand,Price,Seller,Is_Fake,min_price_last_month,sub_category
1273,744601,مداد چشم گرین فیس شماره 103,80,1,آرایش چشم,مداد چشم,گرین فیس,650000,بی اکسلنت,False,0,beauty
1270,744601,مداد چشم گرین فیس شماره 103,80,1,آرایش چشم,مداد چشم,گرین فیس,650000,بی اکسلنت,False,0,beauty
724,763291,مداد ابرو پودری تایرا شماره 406,64,171,آرایش ابرو,مداد ابرو,تایرا,450000,باباریا,False,0,beauty
717,763291,مداد ابرو پودری تایرا شماره 406,64,171,آرایش ابرو,مداد ابرو,تایرا,450000,باباریا,False,0,beauty
1420,978223,مژه مصنوعی 2پوینت مدل 2PA,78,40,آرایش چشم,مژه مصنوعی,2پوینت,550000,پلی فکتوری,False,0,beauty
...,...,...,...,...,...,...,...,...,...,...,...,...
475,11843256,مداد ابرو داویس مدل Extera Waterproof شماره 040,0,0,آرایش ابرو,مداد ابرو,متفرقه,850000,میس مد,False,0,beauty
1082,12249418,مژه مصنوعی آردل مدل 3D-46,0,0,آرایش چشم,مژه مصنوعی,آردل,850000,ایدن گاردن,False,0,beauty
1070,12249418,مژه مصنوعی آردل مدل 3D-46,0,0,آرایش چشم,مژه مصنوعی,آردل,850000,ایدن گاردن,False,0,beauty
927,12547585,صابون لیفت ابرو مک شاین مدل D1,0,0,آرایش ابرو,ژل و صابون ابرو,متفرقه,1100000,پرشین پرفیوم,False,0,beauty
